# Seed runs: C0 to C10 from frozen weights and frozen loss weights

**This is the final entry point for training.** It trains the 11 configurations for one seed and nothing else.

| Frozen input | Value | Source |
|---|---|---|
| Loss weights λ | `LAMBDAS` below | seed-42 calibration (95% perceptual share at initialisation); never recalibrated per seed |
| CT dose network | regression v2, 1 output, sha256 `69feb2a1…1774` | `results/seed42/ct_dose_net_v2_regression/ct_dose_net.pt` (not in git; Drive or manual upload) |
| Feature distance | plain sum over blocks, variance fix | `src/distances.py` |
| Protocol | AdamW 1e-4, batch 8, 30 epochs, `ReduceLROnPlateau(0.5)`, AMP, lowest validation L1, one shared 224×224 crop per batch | `paper_plan.md` |

Seed 42 is **not** retrained here: its rows are final (`results/seed42/`, see `results/provenance.json`). The other two notebooks are records of how those rows were made:
`refactored_perceptual_backbone_ldct.ipynb` (Weeks 1 and 2: checks, screen, calibration, C0 to C7) and
`slides_and_reports/ctdosenet_followup_ldct.ipynb` (variance fix, dose network v2, final C8, C9, C10).

How to use it, from a fresh kernel each time:

1. `SMOKE = True`: wiring check on 8 slices and 1 epoch. Not a result.
2. `SMOKE = False`, `REPRO_CHECK = True`: one epoch of C9 (CT dose network v2, pointwise) at seed 42 is compared with the logged epoch 1 of the final run. If it does not match, stop: the seeds would not be comparable with seed 42.
3. `SEED = 123`, then `SEED = 456`. Copy `outputs/perceptual_backbone_seeds/seed{SEED}/` (without `best.pth`) into `results/`.

The two optional sections at the end (Step 1.3 check of the v2 dose network, replay of C10 with validation diagnostics) are off by default and never touch the test split.

## 0. Run configuration

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

NOTEBOOK_VERSION = "2026-10-13"      # written into every run_summary.json
SEED = 123                           # 123 or 456
SMOKE = False                        # True: 8 slices per split, 1 epoch (wiring check, not a result)
REPRO_CHECK = True                   # one epoch of C9 at seed 42 against the logged epoch 1, before any training
TRAIN_CONFIGS = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10"]
CHECK_DOSE_V2_DISTANCE = False       # optional: Step 1.3 distance check of the v2 dose network
DIAGNOSE_C10 = False                 # optional: replay C10 at seed 42 with per-slice validation diagnostics
EXPORT_FIG3 = False                  # optional: predictions of the Fig 3 slices from the saved checkpoints (no training)

WORK_DIR = Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = Path.cwd() if (Path.cwd() / "src" / "config.py").exists() else WORK_DIR / "perceptual_backbone_ldct"
REPO_URL = "https://github.com/UnkindledTwo/perceptual_backbone_ldct"
DATA_ROOT = WORK_DIR / "minideeplesion"
DATASET_GDRIVE_ID = "1pIUT5beAkp17WCxyGPHZfHy4kEiJtD4y"
OUTPUT_DIR = WORK_DIR / "outputs" / ("perceptual_backbone_seeds_smoke" if SMOKE else "perceptual_backbone_seeds")
DRIVE_DIR = Path("/content/drive/MyDrive/BIAI/ISBI2027_WEIGHTS") if IN_COLAB else None

# Protocol (paper_plan.md). Changing any of these means re-running every row of every seed.
EPOCHS = 1 if SMOKE else 30
BATCH_SIZE = 8
CROP_SIZE = 224
DETERMINISTIC = True
NUM_WORKERS = 0 if SMOKE else 4
FORCE_RERUN = False                  # False: a finished run (run_summary.json present) is loaded, not retrained
RADIMAGENET_STATS = None             # ImageNet statistics, as in the seed-42 runs

# id: (feature network key, matching, label)
CONFIG_SPECS = {
    "C0": ("none", "pointwise", "C0 (L1)"),
    "C1": ("vgg19", "pointwise", "C1 (VGG-19, pointwise)"),
    "C2": ("lpips_vgg", "pointwise", "C2 (LPIPS-VGG, pointwise)"),
    "C3": ("dists", "statistics", "C3 (DISTS, statistics)"),
    "C4": ("radimagenet", "pointwise", "C4 (RadImageNet, pointwise)"),
    "C5": ("dinov2_s", "pointwise", "C5 (DINOv2-S, pointwise)"),
    "C6": ("dinov2_b", "pointwise", "C6 (DINOv2-B, pointwise)"),
    "C7": ("medsiglip", "pointwise", "C7 (MedSigLIP, pointwise)"),
    "C8": ("dinov2_s", "statistics", "C8 (DINOv2-S, statistics)"),
    "C9": ("ct_dose_net", "pointwise", "C9 (CT dose network v2, pointwise)"),
    "C10": ("ct_dose_net", "statistics", "C10 (CT dose network v2, statistics)"),
}

# Frozen loss weights: the `loss_weight` of the final seed-42 run of each configuration.
LAMBDAS = {
    "C0": 0.0,
    "C1": 4.699193400742626,         # results/calibration_weights.csv
    "C2": 9.162468785905386,
    "C3": 6.983633737277652,
    "C4": 2.2106317891201495,
    "C5": 1.6343220755710788,
    "C6": 2.4972450632574033,
    "C7": 3.755348092186327,
    "C8": 1.9659469513086878,        # results/calibration_varfix.csv (after the variance fix)
    "C9": 1.5492900582384366,        # results/calibration_v2.csv (dose network v2)
    "C10": 2.801559888260101,
}

# Frozen CT dose network (regression v2). The file is chosen by its hash, not by its name.
DOSE_NET_SHA256 = "69feb2a1bc7d23eb80f6bc88b57d99ea893f38284fed43876225df4b81581774"
DOSE_NET_CANDIDATES = [p for p in (
    REPO_DIR / "results" / "seed42" / "ct_dose_net_v2_regression" / "ct_dose_net.pt",
    DRIVE_DIR / "ct_dose_net_v2_regression.pt" if DRIVE_DIR else None,
    DRIVE_DIR / "ct_dose_net_v2_regression" / "ct_dose_net.pt" if DRIVE_DIR else None,
    WORK_DIR / "ct_dose_net_v2_regression.pt",
    WORK_DIR / "ct_dose_net.pt",
) if p is not None]

# Epoch 1 of results/seed42/train_C9_v2/train_log.csv, the reference of REPRO_CHECK.
REPRO_REFERENCE = {"train_loss": 2.792584854782092, "val_l1": 9280.567372081045}
REPRO_TOLERANCE = 1e-3               # relative; the runs are deterministic, small differences come from the GPU and library build

assert SEED in (123, 456), "seed 42 is final and is not retrained here (REPRO_CHECK and DIAGNOSE_C10 use it on their own)"

## 1. Setup

In [ ]:
%pip install -q lpips piq huggingface_hub gdown

In [ ]:
if not DATA_ROOT.exists():
    !gdown {DATASET_GDRIVE_ID} -O {WORK_DIR}/dataset_biailab_I2I_2025v1.zip
    !unzip -q -o {WORK_DIR}/dataset_biailab_I2I_2025v1.zip -d {WORK_DIR}
if not (REPO_DIR / "src" / "config.py").exists():
    !git clone {REPO_URL} {REPO_DIR}
if IN_COLAB and DRIVE_DIR is not None:
    from google.colab import drive
    drive.mount("/content/drive")
assert DATA_ROOT.exists(), f"dataset not found at {DATA_ROOT}"
assert (REPO_DIR / "src" / "val_diagnostics.py").exists(), (
    "the checkout is older than this notebook: push src/ to GitHub and delete the stale clone, or upload src/ by hand")

In [ ]:
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import hashlib
import json
import platform
import random
import shutil
import subprocess
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet50
from tqdm.auto import tqdm

from src.config import PerceptualConfig
from src.ct_dose_net import DoseNetFeatures, load_dose_net
from src.data import I2IPairs, assert_no_patient_leakage, patient_id
from src.distances import FeatureDistance, PerceptualObjective, PiqDistance
from src.feature_nets import IMAGENET, FeatureNet, build_feature_net
from src.metrics import slice_metrics
from src.model import UNet
from src.train import validation_l1
from src.val_diagnostics import per_slice_validation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_ROOT = OUTPUT_DIR / f"seed{SEED}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
METRICS = ["psnr", "ssim", "lpips", "dists", "r_h"]
print("device:", device, "| repo:", REPO_DIR, "| outputs:", RUN_ROOT, "| SMOKE:", SMOKE)
if device.type != "cuda":
    print("WARNING: no GPU. Only SMOKE runs are sensible on CPU.")

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = DETERMINISTIC
    torch.backends.cudnn.benchmark = not DETERMINISTIC


def _seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed: int = SEED) -> DataLoader:
    """Loader whose shuffling and worker seeds come from its own generator, not the global RNG."""
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      generator=generator, worker_init_fn=_seed_worker)


def sync() -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()


def count_params(module: nn.Module | None) -> int:
    return 0 if module is None else sum(p.numel() for p in module.parameters())


def free(*modules: nn.Module | None) -> None:
    for module in modules:
        if module is not None:
            module.cpu()
    if device.type == "cuda":
        torch.cuda.empty_cache()


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def git(*arguments: str) -> str:
    try:
        return subprocess.run(["git", "-C", str(REPO_DIR), *arguments], capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""


def source_record() -> dict:
    """Which code this kernel runs: commit, uncommitted source files, and a hash of every module in src/."""
    changed = [line for line in git("status", "--porcelain", "--", "src").splitlines() if "__pycache__" not in line]
    return {"repo_commit": git("rev-parse", "HEAD") or "unknown", "uncommitted_src_files": changed,
            "src_sha256": {path.name: sha256(path) for path in sorted((REPO_DIR / "src").glob("*.py"))},
            "notebook_version": NOTEBOOK_VERSION}

In [ ]:
FILES = {split: sorted((DATA_ROOT / split).glob("*.npz")) for split in ("train", "val", "test")}
patients_per_split = assert_no_patient_leakage(DATA_ROOT)
DATASETS = {split: I2IPairs(DATA_ROOT / split, "noisy_clipped_100k", "gt_clipped") for split in ("train", "val", "test")}
if SMOKE:
    for split in FILES:
        FILES[split] = FILES[split][:8]
        DATASETS[split].files = FILES[split]
else:
    assert {s: len(f) for s, f in FILES.items()} == {"train": 899, "val": 182, "test": 220}, "split sizes differ from the Protocol"
print("patients per split:", patients_per_split, "| slices:", {s: len(f) for s, f in FILES.items()})

## 2. Frozen feature networks

RadImageNet is loaded with the verified loader of the Weeks 1 and 2 notebook (every backbone tensor must come from the checkpoint).
The CT dose network is accepted only if its hash is the frozen regression-v2 file.

In [ ]:
_SEQUENTIAL_TO_RESNET = {"0": "conv1", "1": "bn1", "4": "layer1", "5": "layer2", "6": "layer3", "7": "layer4"}
_WRAPPER_KEYS = ("state_dict", "model_state_dict", "model")
_PREFIXES = ("module.", "backbone.", "model.", "encoder.")


def _is_backbone_tensor(name: str) -> bool:
    return not name.startswith("fc.") and not name.endswith("num_batches_tracked")


def load_radimagenet_state(body: nn.Module, weights_path: Path) -> dict[str, int]:
    raw = torch.load(weights_path, map_location="cpu")
    if isinstance(raw, nn.Module):
        raw = raw.state_dict()
    for wrapper in _WRAPPER_KEYS:
        if isinstance(raw, dict) and isinstance(raw.get(wrapper), dict):
            raw = raw[wrapper]
    expected = body.state_dict()
    state, skipped = {}, []
    for key, value in raw.items():
        name = key
        while name.startswith(_PREFIXES):
            name = name.split(".", 1)[1]
        head, _, rest = name.partition(".")
        if head in _SEQUENTIAL_TO_RESNET and rest:
            name = f"{_SEQUENTIAL_TO_RESNET[head]}.{rest}"
        if name in expected and expected[name].shape == value.shape:
            state[name] = value
        else:
            skipped.append(key)
    result = body.load_state_dict(state, strict=False)
    missing = [name for name in result.missing_keys if _is_backbone_tensor(name)]
    if missing:
        raise RuntimeError(f"RadImageNet weights did not load: {len(missing)} backbone tensors missing, "
                           f"for example {missing[:5]}; checkpoint keys look like {list(raw)[:5]}")
    wanted = sum(_is_backbone_tensor(name) for name in expected)
    loaded = sum(_is_backbone_tensor(name) for name in state)
    print(f"RadImageNet: loaded {loaded} of {wanted} backbone tensors; {len(skipped)} checkpoint keys not used {skipped[:4]}")
    return {"loaded": loaded, "expected": wanted, "skipped": len(skipped)}


class RadImageNetFeatures(FeatureNet):
    """ResNet-50 layer1 to layer3 features with verified RadImageNet weights."""

    def __init__(self, weights_path: Path, stats=None) -> None:
        super().__init__(224, stats or IMAGENET)
        self.body = resnet50(weights=None)
        self.load_report = load_radimagenet_state(self.body, weights_path)

    def features(self, image: torch.Tensor) -> list[torch.Tensor]:
        b = self.body
        x = b.maxpool(b.relu(b.bn1(b.conv1(self.prepare(image)))))
        f1 = b.layer1(x)
        f2 = b.layer2(f1)
        return [f1, f2, b.layer3(f2)]

In [ ]:
from huggingface_hub import hf_hub_download, login

wanted_nets = {CONFIG_SPECS[c][0] for c in TRAIN_CONFIGS}
RADIMAGENET_WEIGHTS = None
if "medsiglip" in wanted_nets:
    login()        # google/medsiglip-448 is a gated model
if "radimagenet" in wanted_nets:
    RADIMAGENET_WEIGHTS = Path(hf_hub_download(repo_id="Lab-Rasool/RadImageNet", filename="ResNet50.pt"))

found = {str(path): sha256(path) for path in DOSE_NET_CANDIDATES if path.exists()}
DOSE_NET_PATH = next((Path(path) for path, digest in found.items() if digest == DOSE_NET_SHA256), None)
assert DOSE_NET_PATH is not None, (
    "the frozen regression-v2 dose network was not found. Upload results/seed42/ct_dose_net_v2_regression/ct_dose_net.pt "
    f"to one of {[str(p) for p in DOSE_NET_CANDIDATES]}. Files seen (path: sha256): {found}")
dose_outputs = load_dose_net(DOSE_NET_PATH).head[-1].out_features
assert dose_outputs == 1, f"expected the 1-output regression network, found {dose_outputs} outputs"
print("dose network:", DOSE_NET_PATH, "| outputs:", dose_outputs)


def make_distance(name: str, matching: str) -> nn.Module | None:
    """The frozen distance of one configuration. Plain sum over blocks for every feature network."""
    if name == "none":
        return None
    if name in ("lpips_vgg", "dists"):
        return PiqDistance(name)
    if name == "ct_dose_net":
        return FeatureDistance(DoseNetFeatures(load_dose_net(DOSE_NET_PATH)), matching)
    if name == "radimagenet":
        return FeatureDistance(RadImageNetFeatures(RADIMAGENET_WEIGHTS, RADIMAGENET_STATS), matching)
    return FeatureDistance(build_feature_net(name), matching)


SOURCE = source_record()
SETTINGS = {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "crop_size": CROP_SIZE, "deterministic": DETERMINISTIC,
            "num_workers": NUM_WORKERS, "smoke": SMOKE, "dose_net_sha256": DOSE_NET_SHA256,
            "radimagenet_stats": "imagenet" if RADIMAGENET_STATS is None else RADIMAGENET_STATS,
            "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
            "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu"}
print("commit:", SOURCE["repo_commit"], "| uncommitted src files:", SOURCE["uncommitted_src_files"] or "none")

## 3. Training loop

The timed loop of the two seed-42 notebooks, unchanged in what it computes. The U-Net is built right after seeding, the feature
network after it, and the seed is set again before the first batch, so the initial U-Net and the crop sequence depend only on the seed.

In [ ]:
WARMUP_STEPS = 10      # first steps left out of the ms-per-step average


def train_run(run_name: str, config_id: str, seed: int = SEED, run_root: Path = RUN_ROOT, epochs: int = EPOCHS,
              evaluate_test: bool = True, epoch_hook=None) -> tuple[pd.DataFrame | None, dict]:
    """Train one configuration with its frozen loss weight, test it once, and record cost and provenance.

    `epoch_hook(model, epoch)` runs after the validation of each epoch (diagnostics only; it must not change the model).
    """
    feature_net, matching, label = CONFIG_SPECS[config_id]
    config = PerceptualConfig(run_name=run_name, data_root=DATA_ROOT, feature_net=feature_net, matching=matching,
                              loss_weight=LAMBDAS[config_id], output_dir=run_root, crop_size=CROP_SIZE, seed=seed,
                              batch_size=BATCH_SIZE, epochs=epochs)
    run_dir = config.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    summary_path, metrics_path, checkpoint = run_dir / "run_summary.json", run_dir / "metrics.csv", run_dir / "best.pth"
    if summary_path.exists() and not FORCE_RERUN:
        print(f"{label}: finished run found in {run_dir}, loading it")
        frame = pd.read_csv(metrics_path, dtype={"patient_id": str}) if metrics_path.exists() else None
        return frame, json.loads(summary_path.read_text())

    seed_everything(seed)
    model = UNet().to(device)
    distance = make_distance(feature_net, matching)
    criterion = PerceptualObjective(distance, config.loss_weight, config.crop_size).to(device)
    seed_everything(seed)

    train_loader = make_loader(DATASETS["train"], config.batch_size, shuffle=True, seed=seed)
    val_loader = make_loader(DATASETS["val"], config.batch_size, seed=seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5)
    use_amp = config.use_amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    best, best_epoch, nonfinite, step_ms, log = float("inf"), -1, 0, [], []
    run_start = time.perf_counter()
    for epoch in tqdm(range(config.epochs), desc=run_name):
        model.train()
        epoch_start, running, seen = time.perf_counter(), 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, (noisy, clean, _) in enumerate(train_loader):
            noisy, clean = noisy.to(device), clean.to(device)
            sync()
            step_start = time.perf_counter()
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                prediction = model(noisy)
            loss = criterion(prediction.float(), clean)       # perceptual term in float32
            if not torch.isfinite(loss):
                nonfinite += 1
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss / config.grad_accumulation).backward()
            if (step + 1) % config.grad_accumulation == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            sync()
            step_ms.append((time.perf_counter() - step_start) * 1e3)
            running += loss.item() * noisy.size(0)
            seen += noisy.size(0)
        score = validation_l1(model, val_loader, device)
        scheduler.step(score)
        if score < best:                                      # lowest validation L1, for every configuration
            best, best_epoch = score, epoch + 1
            torch.save(model.state_dict(), checkpoint)
        log.append({"epoch": epoch + 1, "train_loss": running / max(seen, 1), "val_l1": score,
                    "lr": optimizer.param_groups[0]["lr"], "epoch_seconds": time.perf_counter() - epoch_start})
        if epoch_hook is not None:
            epoch_hook(model, epoch + 1)
    train_seconds = time.perf_counter() - run_start
    pd.DataFrame(log).to_csv(run_dir / "train_log.csv", index=False)

    frame, test_seconds = None, 0.0
    if evaluate_test:
        model.load_state_dict(torch.load(checkpoint, map_location=device))
        model.eval()
        rows, test_start = [], time.perf_counter()
        with torch.no_grad():
            for noisy, clean, stem in DATASETS["test"]:       # the test split is touched once, here
                prediction = model(noisy[None].to(device)).clamp(0, 1)[0, 0].float().cpu().numpy()
                rows.append({"slice_id": stem, "patient_id": patient_id(Path(stem)),
                             **slice_metrics(prediction, clean[0].numpy())})
        test_seconds = time.perf_counter() - test_start
        frame = pd.DataFrame(rows)
        frame.insert(0, "seed", seed)
        frame.insert(0, "configuration", config_id)
        frame.to_csv(metrics_path, index=False)

    timed = step_ms[WARMUP_STEPS:] if len(step_ms) > WARMUP_STEPS else step_ms
    summary = {
        "config": config_id, "label": label, "feature_net": feature_net, "matching": matching,
        "loss_weight": config.loss_weight, "seed": seed, "epochs": config.epochs, "batch_size": config.batch_size,
        "grad_accumulation": config.grad_accumulation,
        "ms_per_step": float(np.mean(timed)) if timed else float("nan"),
        "ms_per_step_median": float(np.median(timed)) if timed else float("nan"),
        "steps": len(step_ms), "nonfinite_steps": nonfinite,
        "train_seconds": train_seconds, "test_seconds": test_seconds, "gpu_hours": (train_seconds + test_seconds) / 3600,
        "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 2 ** 30 if device.type == "cuda" else 0.0,
        "best_val_l1": best, "best_epoch": best_epoch,
        "unet_parameters": count_params(model), "feature_parameters": count_params(distance),
        "best_pth_sha256": sha256(checkpoint), "source": SOURCE, "settings": SETTINGS,
    }
    if frame is not None:
        summary |= {"n_test": len(frame), **{f"{m}_mean": float(frame[m].mean()) for m in METRICS},
                    **{f"{m}_std": float(frame[m].std(ddof=1)) for m in METRICS}}
    summary_path.write_text(json.dumps(summary, indent=2))
    free(model, criterion)
    print(f"{label}: best epoch {best_epoch}, {summary['ms_per_step']:.0f} ms/step, "
          f"{summary['gpu_hours']:.2f} GPU h, {nonfinite} non-finite steps")
    return frame, summary

## 4. Reproduction check against seed 42

One epoch of C9 (CT dose network v2, pointwise) at seed 42, no test split. Its training loss and validation L1 must agree with epoch 1 of
`results/seed42/train_C9_v2/train_log.csv`. A mismatch means this runner, the checkout or the dose-network file differs from what made the
seed-42 rows: do not start the seeds, report it.

In [ ]:
if REPRO_CHECK and not SMOKE:
    repro_root = OUTPUT_DIR / "repro_seed42"
    train_run("repro_C9_epoch1", "C9", seed=42, run_root=repro_root, epochs=1, evaluate_test=False)
    observed = pd.read_csv(repro_root / "repro_C9_epoch1" / "train_log.csv").iloc[0]
    repro = pd.DataFrame([{"quantity": key, "seed-42 log": value, "this kernel": observed[key],
                           "relative difference": abs(observed[key] / value - 1)} for key, value in REPRO_REFERENCE.items()])
    repro.to_csv(repro_root / "repro_check.csv", index=False)
    display(repro)
    assert (repro["relative difference"] < REPRO_TOLERANCE).all(), (
        "epoch 1 of C9 at seed 42 does not match the logged run: stop and report this before training any seed")
    print("reproduction check passed")
else:
    print("reproduction check skipped (SMOKE or REPRO_CHECK = False)")

## 5. Train C0 to C10 for this seed

In [ ]:
RESULTS, SUMMARIES, RUN_STATUS = {}, {}, {}
for config_id in TRAIN_CONFIGS:
    print(f"\n>>> {CONFIG_SPECS[config_id][2]} (lambda {LAMBDAS[config_id]:.4f}, seed {SEED})")
    try:
        RESULTS[config_id], SUMMARIES[config_id] = train_run(f"train_{config_id}", config_id)
        RUN_STATUS[config_id] = "finished"
    except Exception as error:      # keep going, but report the failure exactly as it happened
        RUN_STATUS[config_id] = f"failed: {error!r}"
        print(f"<<< {config_id} FAILED: {error!r}")
        if device.type == "cuda":
            torch.cuda.empty_cache()
display(pd.Series(RUN_STATUS, name="status").to_frame())
assert all(status == "finished" for status in RUN_STATUS.values()), "at least one configuration failed"

In [ ]:
rows = []
for config_id, summary in SUMMARIES.items():
    rows.append({"Configuration": summary["label"], "seed": SEED,
                 **{m: summary[f"{m}_mean"] for m in METRICS}, **{f"{m}_std": summary[f"{m}_std"] for m in METRICS},
                 "ms_per_step": summary["ms_per_step"], "gpu_hours": summary["gpu_hours"], "lambda": summary["loss_weight"],
                 "best_epoch": summary["best_epoch"], "nonfinite_steps": summary["nonfinite_steps"]})
df_seed = pd.DataFrame(rows)
df_seed.to_csv(OUTPUT_DIR / f"table1_seed{SEED}.csv", index=False)
display(df_seed[["Configuration", *METRICS, "ms_per_step", "best_epoch", "nonfinite_steps"]])
print("the _std columns are the spread over the test slices of this seed, not over seeds")

## Optional: Step 1.3 distance check of the v2 dose network

Fills the cells that `results/week1_week2_tables.md` marks as not rerun. Copy `zoo_checks_v2.csv` into `results/` and run `python -m src.tables`.

In [ ]:
def check_distance(distance: nn.Module) -> dict:
    """Step 1.3 checks on a random 64 x 64 input."""
    x = torch.rand(1, 1, 64, 64, device=device)
    with torch.no_grad():
        at_equality = distance(x, x.clone()).item()
    prediction = torch.rand(1, 1, 64, 64, device=device, requires_grad=True)
    distance(prediction, torch.rand(1, 1, 64, 64, device=device)).backward()
    gradient = prediction.grad
    reaches = gradient is not None and bool(torch.isfinite(gradient).all()) and gradient.abs().sum().item() > 0
    return {"distance_at_equality": at_equality, "zero_at_equality": abs(at_equality) < 1e-5,
            "gradient_reaches_prediction": reaches}


def backward_ms(distance: nn.Module, batch: int = 8, size: int = 224, warmup: int = 2, repeats: int = 5) -> float:
    """ms per batch of 8 at 224 x 224, forward and backward through the feature distance."""
    times = []
    for index in range(warmup + repeats):
        prediction = torch.rand(batch, 1, size, size, device=device, requires_grad=True)
        target = torch.rand(batch, 1, size, size, device=device)
        sync()
        start = time.perf_counter()
        distance(prediction, target).backward()
        sync()
        if index >= warmup:
            times.append((time.perf_counter() - start) * 1e3)
    return float(np.mean(times))


if CHECK_DOSE_V2_DISTANCE:
    rows = []
    for matching in ("pointwise", "statistics"):
        distance = make_distance("ct_dose_net", matching).to(device)
        seed_everything(42)
        rows.append({"Network": "CT dose network v2 (regression)", "Matching": matching, **check_distance(distance),
                     "ms_per_batch_of_8": backward_ms(distance), "Parameters": count_params(distance.net.network.blocks),
                     "trained": True})
        free(distance)
    df_checks_v2 = pd.DataFrame(rows)
    df_checks_v2.to_csv(OUTPUT_DIR / "zoo_checks_v2.csv", index=False)
    display(df_checks_v2)

## Optional: replay C10 at seed 42 with validation diagnostics

C10 (CT dose network v2, statistics) had validation L1 spikes (80,138 at epoch 3, 30,478 at epoch 29, about 6,000 otherwise). This replays the
run and records, after every epoch and for every validation slice, the L1, the mean signed error and the L1 with BatchNorm in batch-statistics
mode. Validation split only; the test split is not read. The replayed validation L1 is first compared with the committed log.
See `slides_and_reports/c10_validation_instability.md` for how to read the output.

In [ ]:
if DIAGNOSE_C10 and not SMOKE:
    diag_root, per_epoch = OUTPUT_DIR / "seed42", []

    def record_validation(model: nn.Module, epoch: int) -> None:
        frame = per_slice_validation(model, make_loader(DATASETS["val"], BATCH_SIZE, seed=42), device)
        frame.insert(0, "epoch", epoch)
        per_epoch.append(frame)

    train_run("diag_C10_v2", "C10", seed=42, run_root=diag_root, evaluate_test=False, epoch_hook=record_validation)
    diag_dir = diag_root / "diag_C10_v2"
    if per_epoch:
        pd.concat(per_epoch, ignore_index=True).to_csv(diag_dir / "val_per_slice.csv", index=False)
    slices = pd.read_csv(diag_dir / "val_per_slice.csv")

    replay = pd.read_csv(diag_dir / "train_log.csv")[["epoch", "val_l1"]]
    committed = pd.read_csv(REPO_DIR / "results" / "seed42" / "train_C10_v2" / "train_log.csv")[["epoch", "val_l1"]]
    check = replay.merge(committed, on="epoch", suffixes=("_replay", "_committed"))
    check["relative_difference"] = (check["val_l1_replay"] / check["val_l1_committed"] - 1).abs()
    check.to_csv(diag_dir / "replay_check.csv", index=False)
    print(f"replay against the committed log: largest relative difference {check['relative_difference'].max():.2e}")

    def epoch_summary(group: pd.DataFrame) -> pd.Series:
        ordered = group["l1"].sort_values(ascending=False)
        return pd.Series({"val_l1": group["l1"].mean(), "val_l1_batch_statistics": group["l1_batch_statistics"].mean(),
                          "median_slice_l1": group["l1"].median(), "share_of_worst_10_slices": ordered.head(10).sum() / ordered.sum(),
                          "mean_signed_error": group["signed_error"].mean(),
                          "share_of_l1_from_offset": (group["signed_error"].abs() * 512 * 512).sum() / group["l1"].sum()})

    df_diag = slices.groupby("epoch").apply(epoch_summary).reset_index()
    df_diag.to_csv(diag_dir / "val_per_epoch.csv", index=False)
    display(df_diag)

## Optional: export the Fig 3 predictions

Fig 3 shows LDCT, C0, C1, C3, C5, C7, C10 and NDCT on the test slices closest to the median C0 PSNR. The slices are listed in `results/week3/fig3_slices.csv` (written by `python -m src.seeds_tables` from the logged metrics, not chosen by eye). This cell loads each `best.pth` of this seed from `OUTPUT_DIR`, checks its sha256 against `run_summary.json`, predicts the listed slices exactly as the test loop does, and checks the PSNR against the logged `metrics.csv`. It trains nothing and selects nothing. To run only this cell after the setup cells, set `EXPORT_FIG3 = True`, `REPRO_CHECK = False` and unpack `outputs_seed{SEED}.zip` into `OUTPUT_DIR` first.

In [ ]:
FIG3_CONFIGS = ["C0", "C1", "C3", "C5", "C7", "C10"]

if EXPORT_FIG3 and not SMOKE:
    from skimage.metrics import peak_signal_noise_ratio

    wanted = pd.read_csv(REPO_DIR / "results" / "week3" / "fig3_slices.csv")["slice_id"].tolist()
    fig3_dir = OUTPUT_DIR / "fig3" / f"seed{SEED}"
    fig3_dir.mkdir(parents=True, exist_ok=True)
    samples = {stem: (noisy, clean) for noisy, clean, stem in DATASETS["test"] if stem in wanted}
    assert sorted(samples) == sorted(wanted), "a Fig 3 slice is not in the test split"
    images = {stem: {"LDCT": noisy[0].numpy(), "NDCT": clean[0].numpy()} for stem, (noisy, clean) in samples.items()}
    export = {"seed": SEED, "slices": wanted, "configurations": FIG3_CONFIGS, "source": source_record(), "checkpoints": {}, "psnr": {}}
    for config_id in FIG3_CONFIGS:
        run_dir = RUN_ROOT / f"train_{config_id}"
        summary = json.loads((run_dir / "run_summary.json").read_text())
        assert sha256(run_dir / "best.pth") == summary["best_pth_sha256"], f"{config_id}: best.pth is not the logged checkpoint"
        logged = pd.read_csv(run_dir / "metrics.csv").set_index("slice_id")["psnr"]
        model = UNet().to(device)
        model.load_state_dict(torch.load(run_dir / "best.pth", map_location=device))
        model.eval()
        with torch.no_grad():
            for stem, (noisy, clean) in samples.items():
                prediction = model(noisy[None].to(device)).clamp(0, 1)[0, 0].float().cpu().numpy()
                psnr = float(peak_signal_noise_ratio(clean[0].numpy(), prediction, data_range=1.0))
                assert abs(psnr - logged[stem]) < 1e-3, f"{config_id} {stem}: PSNR {psnr:.4f}, logged {logged[stem]:.4f}"
                images[stem][config_id] = prediction
                export["psnr"][f"{config_id}/{stem}"] = psnr
        export["checkpoints"][config_id] = summary["best_pth_sha256"]
        free(model)
    for stem, arrays in images.items():
        np.savez_compressed(fig3_dir / f"{Path(stem).stem}.npz", **arrays)
    export["files_sha256"] = {path.name: sha256(path) for path in sorted(fig3_dir.glob("*.npz"))}
    (fig3_dir / "fig3_export.json").write_text(json.dumps(export, indent=1))
    print("Fig 3 predictions written to", fig3_dir, "| copy the folder to results/week3/fig3/ in the repository")

In [ ]:
# Optional backup to Google Drive (with the checkpoints).
if DRIVE_DIR is not None and DRIVE_DIR.parent.exists() and not SMOKE:
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    archive = shutil.make_archive(str(DRIVE_DIR / f"outputs_seed{SEED}"), "zip", str(OUTPUT_DIR))
    print("backup written to", archive)
else:
    print("no Drive backup (SMOKE or no Drive folder); outputs stay in", OUTPUT_DIR)